# 03 - The RAG pipeline, step by step

**Question this notebook answers:** what exactly happens between a user's question and the answer?

```
01 Data + KB  ->  02 Build index  ->  03 RAG pipeline  ->  04 Evaluation
                                        (you are here)
```

```
question -> embed -> FAISS top 20 -> rerank, keep top 5 -> enough evidence?
                                                              no  -> abstain (no LLM call)
                                                              yes -> grounded prompt -> one LLM call
                                                                     -> answer + sources + disclaimer
```

Part 1 does every step by hand so you can see what goes in and what comes out.
Part 2 builds the same thing from the package code (`src/pipeline.py`), shows it returns the same passages, and runs a few questions through it.
The API (`app.py`) and the evaluation use that same package code.

**What the pipeline deliberately does not do:** no query classification or routing, no second retrieval pass, no retry with a looser prompt, and no fallback to the model's own knowledge. If the sources are not good enough, it says so.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config
from src.generator import build_messages, generate, make_client
from src.kb import load_encoder, load_index
from src.pipeline import RAGPipeline, format_result
from src.retriever import Retriever, load_reranker

index, chunks = load_index()
encoder = load_encoder()        # embeds the question (the same model that embedded the chunks)
reranker = load_reranker()      # cross-encoder
client = make_client()          # Groq, through the OpenAI SDK
threshold, calibrated = config.load_threshold()

print(f"{index.ntotal:,} chunks | abstain below a rerank score of {threshold} (calibrated: {calibrated})")

## Part 1 - one question, step by step

In [ ]:
question = "What are the symptoms of type 2 diabetes?"

### Step 1 - embed the question and search FAISS

The question becomes a vector with the same model used for the chunks. FAISS returns the 20 chunks whose vectors are closest.
This is fast and finds chunks about the same thing even with different wording, but the ranking is rough: question and chunk were embedded separately and are only compared at the end.

In [ ]:
query_vector = encoder.encode([question], normalize_embeddings=True).astype("float32")
print("query vector:", query_vector.shape)

scores, ids = index.search(query_vector, config.RETRIEVE_K)
candidates = chunks.iloc[ids[0]].copy()
candidates["faiss_score"] = scores[0]
candidates["faiss_rank"] = range(1, len(candidates) + 1)
candidates[["faiss_rank", "faiss_score", "source", "title"]].head(10)

### Step 2 - rerank

The cross-encoder reads the question and one chunk **together** and scores how well the chunk answers the question. It is more accurate but slower, so it only sees these 20 candidates.
The best 5 are kept. Compare their new order with `faiss_rank`.

In [ ]:
pairs = [(question, text) for text in candidates["text"]]
candidates["rerank_score"] = reranker.predict(pairs)

top = candidates.sort_values("rerank_score", ascending=False).head(config.CONTEXT_K)
top[["rerank_score", "faiss_rank", "source", "title"]]

### Step 3 - is there enough evidence?

Retrieval always returns *something*, even for a question the sources know nothing about. The rerank score of the best passage decides whether to answer.
The reranker outputs raw logits: above 0 roughly means "more likely relevant than not". The threshold comes from `src/config.py` until notebook 04 calibrates it on the dev questions.

In [ ]:
best = top["rerank_score"].iloc[0]
print(f"best rerank score: {best:.2f} | threshold: {threshold} | decision: {'answer' if best >= threshold else 'abstain'}")

### Step 4 - grounded prompt

The passages are numbered so the model can cite them as `[1]`, `[2]`. The system prompt tells it to use **only** these passages, to say what they do not cover, and not to give personal medical advice.

In [ ]:
passages = top.to_dict("records")
system_message, user_message = build_messages(question, passages)
print(system_message["content"])
print("=" * 80)
print(user_message["content"][:2500])

### Step 5 - one LLM call

One call, temperature 0. If the model returns nothing, `generate` raises an error instead of returning a blank answer.

In [ ]:
print(generate(client, question, passages))

### When the evidence is not good enough

Same steps, on a question about a condition that does not exist. The answer is not "ask the LLM anyway": the pipeline stops at step 3.

In [ ]:
retriever = Retriever(index, chunks, encoder, reranker)

invented = "What are the symptoms of Zelvarin-Moss syndrome?"
found = retriever.retrieve(invented)
print(f"best rerank score: {found[0]['rerank_score']:.2f} | threshold: {threshold} | decision: {'answer' if found[0]['rerank_score'] >= threshold else 'abstain'}")
print("best chunk anyway:", found[0]["source"], "-", found[0]["title"])

## Part 2 - the same thing as one object

`RAGPipeline` (`src/pipeline.py`) runs steps 1-5 and returns a dictionary. First, a check that it retrieves the same 5 passages as the manual steps:

In [ ]:
pipeline = RAGPipeline(retriever, client, threshold, calibrated)

assert {p["chunk_id"] for p in retriever.retrieve(question)} == set(top["chunk_id"])
print("same top-5 chunks as the manual steps")

`format_result` prints the answer, the numbered sources (only the passages that were in the prompt) and the disclaimer.
The questions below show different behaviours: a normal one, a vague one, a plain one, an invented syndrome (should abstain), a possible emergency, and a request for a dose (should not give one).
Read each answer critically: do the `[n]` citations match the sources, and does the answer stay inside what the passages say?

In [ ]:
demo_questions = [
    "What are the symptoms of type 2 diabetes?",
    "My 3 year old has a fever and is very sleepy. What should I do?",
    "How is high blood pressure usually treated?",
    "What are the symptoms of Zelvarin-Moss syndrome?",
    "I have crushing chest pain that spreads to my left arm. What should I do?",
    "How many mg of ibuprofen should I give my 4 year old?",
]

for q in demo_questions:
    result = pipeline.answer(q)
    print("=" * 80)
    print("Q:", q)
    print(f"abstained: {result['abstained']} | best rerank score: {result['top_score']:.2f} | {result['latency_ms'] / 1000:.1f} s\n")
    print(format_result(result))

## Where the code lives

| Step | File | Function |
|---|---|---|
| Embed + FAISS top 20 | `src/retriever.py` | `Retriever.search` |
| Rerank, keep top 5 | `src/retriever.py` | `Retriever.rerank` |
| Evidence check, orchestration | `src/pipeline.py` | `RAGPipeline.answer` |
| Grounded prompt | `src/generator.py` | `SYSTEM_PROMPT`, `build_messages` |
| One LLM call | `src/generator.py` | `generate` |
| Settings (models, k, threshold) | `src/config.py` | |
| HTTP API | `app.py` | `POST /query`, `GET /health` |

**Next:** `04_evaluation.ipynb` measures retrieval, groundedness and safety behaviour against an LLM-only baseline, and calibrates the threshold used in step 3.